In [2]:
# =============================================================================
# v5.1 SAFE MASTER PIPELINE
# Builds on v5 with:
# - Safer metric-to-construct mapping
# - Deterministic PubMed query planning
# - Plain M10 interpreted as activity, not light, unless metric name says "light"
# - Safer phrase-level clinical safety cleaner
# - No automatic diagnosis
# - No automatic treatment recommendation
# - Strict PMID sanitization
# - Filtered evidence used for literature synthesis
# =============================================================================

from __future__ import annotations

import json
import re
import sqlite3
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal, TypedDict

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END


# =============================================================================
# Block I — Locate project root and import project tools
# =============================================================================

PROJECT_ROOT = Path.cwd()

for _ in range(6):
    if (PROJECT_ROOT / "tools" / "llm_conversation.py").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(
        "Could not locate project root containing tools/llm_conversation.py"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tools.llm_conversation as _lc
from tools.llm_conversation import _compact_report_for_llm
from tools.pubmed_search import RetrievalConfig, evidence_to_text, search_pubmed

print("Imports OK")
print("Project root:", PROJECT_ROOT)


# =============================================================================
# Block II — Main configuration
# =============================================================================

DB_PATH = PROJECT_ROOT / "Actigraph_record.db"

AUDIENCE: Literal["expert", "doctor", "layperson"] = "doctor"

ANAMNESIS = ""

ROW_KEY: tuple[str, str, str] | None = None

AGENT_CONFIG = {
    "data_summariser": {
        "model": "phi4:14b",
        "temperature": 0.1,
    },
    "relevance_judge": {
        "model": "phi4:14b",
        "fallback_model": "qwen3.5:latest",
        "temperature": 0.0,
    },
    "literature_synthesiser": {
        "model": "phi4:14b",
        "fallback_model": "qwen3.5:latest",
        "temperature": 0.1,
    },
    "symptom_metric_linker": {
        "model": "phi4:14b",
        "temperature": 0.1,
    },
    "report_writer": {
        "model": "phi4:14b",
        "fallback_model": "qwen3.5:4b",
        "temperature": 0.1,
    },
}

RETRIEVAL_CFG = RetrievalConfig(
    retmax_per_query=25,
    keep_per_query=8,
    max_total_items=25,
    years_back=15,
    humans_only=True,
    adults_only=(AUDIENCE == "doctor"),
)

# This is relative change, not a statistical z-score.
RELATIVE_CHANGE_THRESHOLD = 0.20

MAX_CONSTRUCTS = 6
MIN_ITEMS_AFTER_JUDGE = 8

THREAD_ID = "llm-pipeline-v5-1-safe"


# =============================================================================
# Block III — Load latest analysis row from SQLite
# =============================================================================

con = sqlite3.connect(DB_PATH)

rows = con.execute(
    "SELECT username, period_id_1, period_id_2, audience, model, created_at "
    "FROM ai_analysis_runs ORDER BY created_at DESC"
).fetchall()

if not rows:
    raise RuntimeError("No rows found in ai_analysis_runs")

if ROW_KEY is None:
    ROW_KEY = (rows[0][0], rows[0][1], rows[0][2])

print(f"\nSelected: username={ROW_KEY[0]} P1={ROW_KEY[1]} P2={ROW_KEY[2]}")

raw = con.execute(
    "SELECT json_input FROM ai_analysis_runs "
    "WHERE username=? AND period_id_1=? AND period_id_2=? "
    "ORDER BY created_at DESC LIMIT 1",
    ROW_KEY,
).fetchone()

con.close()

if raw is None:
    raise RuntimeError("No json_input found for selected row")

report_data = json.loads(raw[0])
compact_report = _compact_report_for_llm(report_data)


# =============================================================================
# Block IV — Safe clinical metric rules
# =============================================================================

@dataclass(frozen=True)
class MetricRule:
    down_construct: str
    up_construct: str
    down_interpretation: str
    up_interpretation: str
    down_query: str
    up_query: str
    forbidden_terms: tuple[str, ...] = ()


METRIC_RULES: dict[str, MetricRule] = {
    "Sleep Duration": MetricRule(
        down_construct="short sleep duration",
        up_construct="longer sleep duration",
        down_interpretation=(
            "Estimated sleep duration decreased. This may reflect reduced sleep opportunity, "
            "sleep fragmentation, environmental disturbance, symptoms, medication effects, "
            "or recording artefact. It should not be interpreted alone as circadian amplitude reduction."
        ),
        up_interpretation=(
            "Estimated sleep duration increased. This may reflect longer sleep opportunity, "
            "recovery sleep, reduced fragmentation, or changes in recording context."
        ),
        down_query="short sleep duration daytime function adults",
        up_query="long sleep duration health outcomes adults",
        forbidden_terms=(
            "dampened circadian amplitude",
            "severe circadian disruption",
            "circadian amplitude reduction",
        ),
    ),

    "CPD mid sleep": MetricRule(
        down_construct="greater alignment of mid-sleep timing",
        up_construct="greater deviation of mid-sleep timing",
        down_interpretation=(
            "CPD mid-sleep decreased, suggesting that mid-sleep timing became closer "
            "to the reference pattern."
        ),
        up_interpretation=(
            "CPD mid-sleep increased, suggesting greater deviation of mid-sleep timing "
            "from the reference pattern. This may be compatible with sleep-wake timing "
            "instability or circadian misalignment, but it does not diagnose delayed "
            "sleep-wake phase disorder."
        ),
        down_query="sleep timing regularity circadian alignment adults",
        up_query="sleep timing irregularity circadian misalignment adults",
        forbidden_terms=(
            "delayed sleep phase disorder",
            "DSPD",
            "advanced sleep phase disorder",
            "delayed sleep-wake phase disorder",
        ),
    ),

    "SRI": MetricRule(
        down_construct="sleep irregularity",
        up_construct="sleep regularity",
        down_interpretation=(
            "Sleep regularity decreased, suggesting a less stable sleep-wake pattern "
            "across days."
        ),
        up_interpretation=(
            "Sleep regularity increased, suggesting a more stable sleep-wake pattern "
            "across days."
        ),
        down_query="sleep irregularity health outcomes adults",
        up_query="sleep regularity health outcomes adults",
    ),

    "IS": MetricRule(
        down_construct="reduced interdaily stability",
        up_construct="higher interdaily stability",
        down_interpretation=(
            "Interdaily stability decreased, suggesting weaker day-to-day regularity "
            "of the rest-activity rhythm."
        ),
        up_interpretation=(
            "Interdaily stability increased, suggesting a more regular day-to-day "
            "rest-activity rhythm."
        ),
        down_query="interdaily stability rest activity rhythm health adults",
        up_query="interdaily stability circadian rhythm adults",
    ),

    "IV": MetricRule(
        down_construct="reduced intradaily fragmentation",
        up_construct="increased intradaily fragmentation",
        down_interpretation=(
            "Intradaily variability decreased, suggesting a less fragmented "
            "rest-activity rhythm."
        ),
        up_interpretation=(
            "Intradaily variability increased, suggesting a more fragmented "
            "rest-activity rhythm."
        ),
        down_query="intradaily variability actigraphy fragmentation adults",
        up_query="intradaily variability rest activity fragmentation adults",
    ),

    "RA": MetricRule(
        down_construct="reduced rest-activity amplitude",
        up_construct="stronger rest-activity amplitude",
        down_interpretation=(
            "Relative amplitude decreased, suggesting a weaker contrast between "
            "active and rest periods."
        ),
        up_interpretation=(
            "Relative amplitude increased, suggesting a stronger contrast between "
            "active and rest periods."
        ),
        down_query="relative amplitude actigraphy rest activity rhythm adults",
        up_query="relative amplitude circadian rhythm actigraphy adults",
    ),

    "L5": MetricRule(
        down_construct="lower activity during least active period",
        up_construct="higher activity during least active period",
        down_interpretation=(
            "L5 decreased, suggesting lower activity during the least active period."
        ),
        up_interpretation=(
            "L5 increased, suggesting more activity during the least active period. "
            "Depending on context, this may indicate rest disruption or reduced sleep consolidation."
        ),
        down_query="L5 actigraphy rest activity rhythm adults",
        up_query="nighttime activity sleep fragmentation actigraphy adults",
    ),

    # Important:
    # Plain M10 is interpreted as activity/rest-activity.
    # Only M10 metrics containing "light" are interpreted as light exposure.
    "M10": MetricRule(
        down_construct="lower activity during most active period",
        up_construct="higher activity during most active period",
        down_interpretation=(
            "M10 decreased, suggesting lower activity during the most active period."
        ),
        up_interpretation=(
            "M10 increased, suggesting higher activity during the most active period. "
            "This should be interpreted as a rest-activity feature, not as hyperactivity "
            "or psychiatric disease."
        ),
        down_query="low daytime activity actigraphy adults",
        up_query="daytime activity rest activity rhythm adults",
        forbidden_terms=(
            "ADHD",
            "anxiety",
            "hyperactivity",
            "psychiatric disease",
        ),
    ),

    "SE": MetricRule(
        down_construct="lower sleep efficiency",
        up_construct="higher sleep efficiency",
        down_interpretation=(
            "Sleep efficiency decreased, suggesting poorer sleep continuity or more wake time "
            "during the sleep period."
        ),
        up_interpretation=(
            "Sleep efficiency increased, suggesting improved sleep continuity."
        ),
        down_query="sleep efficiency actigraphy health adults",
        up_query="sleep efficiency actigraphy adults",
    ),

    "WASO": MetricRule(
        down_construct="lower wake after sleep onset",
        up_construct="higher wake after sleep onset",
        down_interpretation=(
            "WASO decreased, suggesting less wake time after sleep onset."
        ),
        up_interpretation=(
            "WASO increased, suggesting more wake time after sleep onset and possible "
            "sleep fragmentation."
        ),
        down_query="wake after sleep onset sleep fragmentation adults",
        up_query="wake after sleep onset actigraphy sleep fragmentation adults",
    ),

    "Mesor": MetricRule(
        down_construct="lower rhythm-adjusted mean level",
        up_construct="higher rhythm-adjusted mean level",
        down_interpretation=(
            "Mesor decreased, suggesting a lower rhythm-adjusted mean level for this signal. "
            "This depends on the signal type and should be interpreted with the original data context."
        ),
        up_interpretation=(
            "Mesor increased, suggesting a higher rhythm-adjusted mean level for this signal. "
            "This depends on the signal type and should be interpreted with the original data context."
        ),
        down_query="cosinor mesor rest activity rhythm adults",
        up_query="cosinor mesor rest activity rhythm adults",
    ),

    "Acrophase": MetricRule(
        down_construct="earlier estimated rhythm phase",
        up_construct="later estimated rhythm phase",
        down_interpretation=(
            "Acrophase decreased, suggesting an earlier estimated timing of the fitted rhythm peak. "
            "This should be interpreted cautiously because acrophase depends on signal type and model fit."
        ),
        up_interpretation=(
            "Acrophase increased, suggesting a later estimated timing of the fitted rhythm peak. "
            "This should be interpreted cautiously because acrophase depends on signal type and model fit."
        ),
        down_query="actigraphy acrophase circadian rhythm adults",
        up_query="actigraphy acrophase circadian rhythm adults",
    ),
}


def find_metric_rule(metric_name: str) -> tuple[str | None, MetricRule | None]:
    """
    Finds the safest matching rule for a metric name.
    Longer keys are checked first so specific names like 'CPD mid sleep'
    are matched before broader terms.
    """
    name = metric_name.lower()

    for key in sorted(METRIC_RULES.keys(), key=len, reverse=True):
        if key.lower() in name:
            return key, METRIC_RULES[key]

    return None, None


def derive_constructs(
    compact_report_text: str,
    relative_threshold: float = RELATIVE_CHANGE_THRESHOLD,
) -> list[dict[str, Any]]:
    """
    Deterministically extracts major metric changes and maps them to safe constructs.

    Important:
    - Uses relative_change, not statistical z-score.
    - CPD is treated as deviation, not diagnosis.
    - Plain M10 is activity/rest-activity.
    - M10 containing "light" is light exposure.
    """
    out: list[dict[str, Any]] = []

    pattern = r"([^|]+?)\s*:\s*P1=([-\d\.]+),\s*P2=([-\d\.]+)"
    matches = re.findall(pattern, compact_report_text)

    for metric_name, p1_str, p2_str in matches:
        metric_name = metric_name.strip()

        try:
            p1 = float(p1_str)
            p2 = float(p2_str)
        except ValueError:
            continue

        if p1 == 0:
            continue

        relative_change = (p2 - p1) / abs(p1)

        if abs(relative_change) < relative_threshold:
            continue

        key, rule = find_metric_rule(metric_name)

        if rule is None:
            continue

        direction = "decreased" if relative_change < 0 else "increased"

        if relative_change < 0:
            construct = rule.down_construct
            interpretation = rule.down_interpretation
            query = rule.down_query
        else:
            construct = rule.up_construct
            interpretation = rule.up_interpretation
            query = rule.up_query

        # Special case: only interpret as light exposure when metric name explicitly says light.
        if key == "M10" and "light" in metric_name.lower():
            if relative_change > 0:
                construct = "higher daytime light exposure"
                interpretation = (
                    "M10 light exposure increased, suggesting higher light exposure during "
                    "the main daytime window. This should be interpreted as light exposure, "
                    "not hyperactivity or psychiatric disease."
                )
                query = "daytime light exposure circadian rhythm sleep adults"
            else:
                construct = "lower daytime light exposure"
                interpretation = (
                    "M10 light exposure decreased, suggesting lower light exposure during "
                    "the main daytime window."
                )
                query = "low daytime light exposure circadian rhythm sleep adults"

        out.append(
            {
                "metric": metric_name,
                "p1": p1,
                "p2": p2,
                "direction": direction,
                "relative_change": round(relative_change, 2),
                "construct": construct,
                "safe_interpretation": interpretation,
                "pubmed_query": query,
                "forbidden_terms": list(rule.forbidden_terms),
            }
        )

    out.sort(key=lambda x: abs(x["relative_change"]), reverse=True)

    return out[:MAX_CONSTRUCTS]


# =============================================================================
# Block V — Safer prompts
# =============================================================================

AGENT1_SYSTEM = """
You are a cautious actigraphy data summariser.

You summarise wearable-derived and actigraphy-derived metrics for downstream clinical interpretation.

Hard rules:
- Do not diagnose any disorder.
- Do not recommend treatment.
- Do not claim that a single wearable metric proves circadian disruption.
- Do not claim that reduced sleep duration proves dampened circadian amplitude.
- Do not claim that CPD proves delayed sleep phase disorder.
- Do not interpret M10 light exposure as hyperactivity.
- Use cautious language: may suggest, is compatible with, warrants review.
- Keep the summary short and patient-specific.
"""

RELEVANCE_SYSTEM = """
You are a PubMed evidence relevance judge for clinical actigraphy reports.

You receive patient-specific actigraphy findings and PubMed evidence items.

Your task:
- Select PMIDs that are relevant to the detected construct.
- Prefer human adult studies, clinical reviews, and studies linking sleep/circadian/rest-activity patterns to outcomes.
- Do not select papers only because they mention a disease name.
- Do not infer diagnosis or treatment from the paper.

Output valid JSON only:
{
  "keep_pmids": ["12345678", "23456789"]
}
"""

AGENT4_SYSTEM = """
You are a clinical literature synthesiser writing for a busy doctor.

You receive safe clinical constructs and matched PubMed evidence.

For EACH construct, write ONE short paragraph, maximum 60 words.

Hard rules:
- Do not diagnose.
- Do not recommend treatment.
- Do not claim causality unless the evidence clearly supports it.
- Do not convert wearable-derived metrics into disease labels.
- Do not interpret light exposure as hyperactivity.
- Do not interpret CPD as delayed sleep phase disorder.
- Use cautious language.

Format exactly:
**Construct name** — paragraph text (PMID xxxxxxxx).

If no direct evidence is matched:
**Construct name** — No direct evidence was retrieved; interpret cautiously.
"""

AGENT6_SYSTEM = _lc._AGENT6_SYSTEM

AGENT5_AUDIENCES = dict(getattr(_lc, "_AGENT5_AUDIENCES", {}))

AGENT5_AUDIENCES["doctor"] = """
You are a clinical data translator writing a cautious actigraphy-based report for a doctor.

Your job is to translate wearable-derived metrics into clinically useful observations.

CRITICAL SAFETY RULES:
1. Do NOT diagnose delayed sleep phase disorder, advanced sleep phase disorder, insomnia, ADHD, anxiety, depression, or any other disorder.
2. Do NOT recommend treatments such as melatonin, bright light therapy, medication, or psychotherapy.
3. Do NOT claim that reduced sleep duration proves circadian disruption.
4. Do NOT claim that reduced sleep duration proves dampened circadian amplitude.
5. Do NOT claim that CPD mid-sleep proves delayed sleep phase disorder.
6. Do NOT interpret light exposure as hyperactivity.
7. Use cautious wording: "may suggest", "is compatible with", "warrants review", "should be interpreted with clinical context".
8. Every key finding must start with the patient's actual P1 and P2 values.
9. If clinical context is missing, explicitly say that interpretation is limited.
10. Cite only PMIDs provided in the literature synthesis. If no PMID is relevant, omit PMID.

You MUST output EXACTLY this format:

## Bottom Line
[One cautious sentence summarizing the main patient-specific pattern. Do not diagnose.]

## Key Findings & Literature Context
- **[Metric Name] ([exact P1 → P2]):** [Patient-specific interpretation using cautious wording]. [One short literature-supported context sentence if available] (PMID xxxxxxxx).
- **[Metric Name] ([exact P1 → P2]):** [Repeat for up to 3 total findings.]

## Suggested Clinical Checks
- [Check recording validity, wear time, artefacts, or whether this was a single-night change.]
- [Review sleep diary, symptoms, medication, pain, hospital routine, naps, and light exposure.]
- [Consider clinical or circadian/sleep-specialist review if the pattern persists.]
"""

AGENT5_AUDIENCES["expert"] = AGENT5_AUDIENCES["doctor"]

AGENT5_AUDIENCES["layperson"] = """
You are writing a cautious patient-facing explanation of wearable-derived sleep and circadian metrics.

Hard rules:
- Do not diagnose.
- Do not recommend treatment.
- Do not mention medication or melatonin.
- Explain that wearable results should be discussed with the clinical team.
- Keep the language simple.

Use this exact format:

## Bottom Line
[One simple cautious sentence.]

## Main Findings
- **[Metric Name] ([P1 → P2]):** [Simple explanation.]

## What to Check With the Clinical Team
- [One practical check.]
- [One practical check.]
- [One practical check.]
"""


# =============================================================================
# Block VI — State and utility functions
# =============================================================================

class PipelineState(TypedDict, total=False):
    compact_report: str
    audience: str
    anamnesis: str

    data_summary: str
    constructs: list[dict[str, Any]]
    search_queries: list[dict[str, str]]

    evidence_items: list[dict[str, Any]]
    raw_abstracts: str
    pmid_list: list[str]

    lit_summary: str
    symptom_metric_table: str
    final_report: str

    trace: list[dict[str, Any]]


def chat_node(
    node: str,
    system: str,
    user: str,
    *,
    json_mode: bool = False,
) -> str:
    cfg = AGENT_CONFIG[node]

    model_names = [cfg["model"]]

    if cfg.get("fallback_model"):
        model_names.append(cfg["fallback_model"])

    last_error: Exception | None = None

    for model_name in model_names:
        try:
            kwargs: dict[str, Any] = {
                "model": model_name,
                "temperature": cfg.get("temperature", 0.2),
            }

            if json_mode:
                kwargs["format"] = "json"

            llm = ChatOllama(**kwargs)

            response = llm.invoke(
                [
                    SystemMessage(content=system),
                    HumanMessage(content=user),
                ]
            )

            return str(response.content).strip()

        except Exception as exc:
            last_error = exc
            print(f"[WARN] {node} failed with {model_name}: {exc}")

    raise RuntimeError(f"{node} failed") from last_error


def safe_json_loads(raw: str) -> dict[str, Any]:
    try:
        return json.loads(raw)
    except Exception:
        return {}


def strict_pmid_sanitizer(report: str, allowed_pmids: set[str]) -> str:
    """
    Removes any PMID citation that is not in the allowed PMID set.
    """
    allowed_pmids = {str(p) for p in allowed_pmids if p}

    def replace_pmid(match: re.Match) -> str:
        pmid = match.group(1)
        if pmid in allowed_pmids:
            return f"(PMID {pmid})"
        return ""

    report = re.sub(
        r"\(?PMID\s*:?\s*(\d{6,10})\)?",
        replace_pmid,
        report,
        flags=re.IGNORECASE,
    )

    report = re.sub(r"\(\s*\)", "", report)
    report = re.sub(r"[ \t]{2,}", " ", report)
    report = re.sub(r"\n{3,}", "\n\n", report)

    return report.strip()


def cited_pmids_in_report(report: str) -> list[str]:
    return sorted(
        set(
            re.findall(
                r"PMID\s*:?\s*(\d{6,10})",
                report,
                flags=re.IGNORECASE,
            )
        )
    )


# =============================================================================
# Block VII — Safer final clinical safety cleaner
# =============================================================================
# Important:
# This cleaner replaces risky assertive phrases only.
# It does NOT replace safe negated phrases like:
# "does not diagnose delayed sleep-wake phase disorder"
# or "not hyperactivity".

RISKY_PHRASE_REPLACEMENTS = [
    (
        r"\b(indicates|suggests|shows|proves)\s+severe circadian disruption\b",
        "may suggest a marked change in the sleep-wake pattern",
    ),
    (
        r"\b(indicates|suggests|shows|proves)\s+(a\s+)?dampened circadian amplitude\b",
        "may suggest altered rest-activity patterning",
    ),
    (
        r"\b(indicates|suggests|shows|proves|is characteristic of)\s+"
        r"(delayed sleep phase disorder|delayed sleep-wake phase disorder|DSPD)\b",
        "may be compatible with sleep timing delay or instability",
    ),
    (
        r"\b(indicates|suggests|shows|proves|is characteristic of)\s+"
        r"(advanced sleep phase disorder)\b",
        "may be compatible with sleep timing advance or instability",
    ),
    (
        r"\b(indicates|suggests|shows|proves|is linked to|is consistent with)\s+"
        r"(ADHD|anxiety disorders?|hyperactivity)\b",
        "cannot be used alone to infer psychiatric disease or hyperactivity",
    ),
    (
        r"\bconsider implementing bright light therapy and melatonin\b",
        "consider clinical review of sleep timing, light exposure, and possible circadian interventions if appropriate",
    ),
    (
        r"\bimplement bright light therapy and melatonin\b",
        "review sleep timing, light exposure, and possible circadian interventions if clinically appropriate",
    ),
]


AWKWARD_SAFETY_REPAIRS = {
    "but it does not diagnose possible sleep timing delay or instability": (
        "but it does not diagnose delayed sleep-wake phase disorder"
    ),
    "not higher measured activity or exposure, depending on the metric": (
        "not hyperactivity"
    ),
    "It should not be interpreted alone as altered rest-activity pattern": (
        "It should not be interpreted alone as circadian amplitude reduction"
    ),
    "It should not be interpreted alone as altered rest-activity pattern.": (
        "It should not be interpreted alone as circadian amplitude reduction."
    ),
}


def clinical_safety_clean(report: str) -> str:
    """
    Deterministic final safety cleaner.

    Important:
    - Replaces only risky assertive phrases.
    - Does not replace terms when they appear in safe negated contexts.
    """
    cleaned = report

    for old, new in AWKWARD_SAFETY_REPAIRS.items():
        cleaned = cleaned.replace(old, new)

    for pattern, replacement in RISKY_PHRASE_REPLACEMENTS:
        cleaned = re.sub(
            pattern,
            replacement,
            cleaned,
            flags=re.IGNORECASE,
        )

    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    return cleaned.strip()


def ensure_required_sections(report: str) -> str:
    """
    Ensures the report has the desired headings.
    This is a fallback if the LLM partially ignores the format.
    """
    required = [
        "## Bottom Line",
        "## Key Findings & Literature Context",
        "## Suggested Clinical Checks",
    ]

    if all(section in report for section in required):
        return report

    repaired = report.strip()

    if "## Suggested Next Steps" in repaired:
        repaired = repaired.replace(
            "## Suggested Next Steps",
            "## Suggested Clinical Checks",
        )

    if not repaired.startswith("## Bottom Line"):
        repaired = "## Bottom Line\n" + repaired

    return repaired


# =============================================================================
# Block VIII — LangGraph nodes
# =============================================================================

def data_summariser(state: PipelineState) -> PipelineState:
    user = (
        "# Actigraphy Metrics\n"
        f"{state['compact_report']}\n\n"
        "Summarise these metrics cautiously for a downstream clinical researcher. "
        "Do not diagnose and do not recommend treatment."
    )

    summary = chat_node(
        "data_summariser",
        AGENT1_SYSTEM,
        user,
    )

    return {
        **state,
        "data_summary": summary,
    }


def construct_mapper(state: PipelineState) -> PipelineState:
    constructs = derive_constructs(
        state["compact_report"],
        relative_threshold=RELATIVE_CHANGE_THRESHOLD,
    )

    print("\nDetected safe constructs:")

    if not constructs:
        print("- No major mapped metric changes detected.")

    for c in constructs:
        print(
            f"- {c['metric']}: {c['p1']} → {c['p2']} "
            f"({c['direction']}, relative_change={c['relative_change']}) "
            f"→ {c['construct']}"
        )

    return {
        **state,
        "constructs": constructs,
    }


def pubmed_query_planner(state: PipelineState) -> PipelineState:
    """
    Deterministic query planner.
    The LLM is not allowed to invent PubMed queries from scratch.
    """
    constructs = state.get("constructs", [])[:MAX_CONSTRUCTS]

    if not constructs:
        queries = [
            {
                "topic": "circadian rhythm disruption",
                "population": "humans",
                "context": "clinical outcomes",
                "expected_link": "general circadian disruption",
            }
        ]

        return {
            **state,
            "search_queries": queries,
        }

    queries: list[dict[str, str]] = []

    for c in constructs:
        queries.append(
            {
                "topic": str(c.get("pubmed_query", c["construct"])),
                "population": "humans",
                "context": "clinical outcomes",
                "expected_link": str(c["construct"]),
            }
        )

    print("\nDeterministic PubMed queries:")

    for q in queries:
        print(f"- {q['topic']}")

    return {
        **state,
        "search_queries": queries,
    }


def pubmed_retriever(state: PipelineState) -> PipelineState:
    items = search_pubmed(
        state.get("search_queries", []),
        config=RETRIEVAL_CFG,
    )

    if len(items) < MIN_ITEMS_AFTER_JUDGE:
        fallback_queries = [
            {
                "topic": c["construct"],
                "population": "humans",
                "context": "clinical outcomes",
                "expected_link": c["construct"],
            }
            for c in state.get("constructs", [])
        ]

        if fallback_queries:
            items += search_pubmed(
                fallback_queries,
                config=RETRIEVAL_CFG,
            )

    # Deduplicate by PMID.
    seen: set[str] = set()
    deduped: list[dict[str, Any]] = []

    for item in items:
        pmid = str(item.get("pmid", ""))

        if not pmid or pmid in seen:
            continue

        seen.add(pmid)
        deduped.append(item)

    return {
        **state,
        "evidence_items": deduped,
        "raw_abstracts": evidence_to_text(deduped),
    }


def relevance_judge(state: PipelineState) -> PipelineState:
    items = state.get("evidence_items", [])

    if not items:
        return {
            **state,
            "pmid_list": [],
            "raw_abstracts": "",
        }

    user = (
        "# Safe clinical constructs\n"
        f"{json.dumps(state.get('constructs', []), ensure_ascii=False, indent=2)}\n\n"
        "# Cautious data summary\n"
        f"{state.get('data_summary', '')}\n\n"
        "# Evidence items\n"
        f"{json.dumps(items, ensure_ascii=False, indent=2)}"
    )

    raw = chat_node(
        "relevance_judge",
        RELEVANCE_SYSTEM,
        user,
        json_mode=True,
    )

    parsed = safe_json_loads(raw)

    keep_pmids = {
        str(x)
        for x in (
            parsed.get("keep_pmids")
            or parsed.get("pmids")
            or []
        )
    }

    if keep_pmids and len(keep_pmids) >= MIN_ITEMS_AFTER_JUDGE:
        filtered_items = [
            x for x in items
            if str(x.get("pmid")) in keep_pmids
        ]
    else:
        filtered_items = items[: max(MIN_ITEMS_AFTER_JUDGE, min(len(items), 18))]

    pmid_list = [
        str(x.get("pmid"))
        for x in filtered_items
        if x.get("pmid")
    ]

    return {
        **state,
        "evidence_items": filtered_items,
        "pmid_list": pmid_list,
        "raw_abstracts": evidence_to_text(filtered_items),
    }


def literature_synthesiser(state: PipelineState) -> PipelineState:
    filtered_abstracts = evidence_to_text(
        state.get("evidence_items", [])
    )

    user = (
        "# Safe Clinical Constructs\n"
        f"{json.dumps(state.get('constructs', [])[:MAX_CONSTRUCTS], ensure_ascii=False, indent=2)}\n\n"
        "# Filtered PubMed Evidence\n"
        f"{filtered_abstracts}\n\n"
        "Write ONE short paragraph per construct. "
        "Do not diagnose. Do not recommend treatment. "
        "Only describe what the wearable-derived pattern may suggest."
    )

    summary = chat_node(
        "literature_synthesiser",
        AGENT4_SYSTEM,
        user,
    )

    return {
        **state,
        "raw_abstracts": filtered_abstracts,
        "lit_summary": summary,
    }


def symptom_metric_linker(state: PipelineState) -> PipelineState:
    if not state.get("anamnesis", "").strip():
        return state

    user = (
        "# Patient Anamnesis\n"
        f"{state['anamnesis']}\n\n"
        "# Data Summary\n"
        f"{state.get('data_summary', '')}\n\n"
        "# Safe Constructs\n"
        f"{json.dumps(state.get('constructs', []), ensure_ascii=False, indent=2)}\n\n"
        "Produce the correlation table cautiously. "
        "Do not diagnose and do not recommend treatment."
    )

    table = chat_node(
        "symptom_metric_linker",
        AGENT6_SYSTEM,
        user,
    )

    return {
        **state,
        "symptom_metric_table": table,
    }


def report_writer(state: PipelineState) -> PipelineState:
    audience = state.get("audience", "doctor")

    system_prompt = AGENT5_AUDIENCES.get(
        audience,
        AGENT5_AUDIENCES["doctor"],
    )

    constructs_block = json.dumps(
        state.get("constructs", [])[:MAX_CONSTRUCTS],
        ensure_ascii=False,
        indent=2,
    )

    user = (
        "# Actigraphy Context\n"
        f"{state['compact_report']}\n\n"
        "# Safe Construct Interpretations\n"
        f"{constructs_block}\n\n"
        "# Literature Synthesis\n"
        f"{state.get('lit_summary', '')}\n\n"
    )

    if state.get("symptom_metric_table"):
        user += (
            "# Symptom-Metric Link Table\n"
            f"{state['symptom_metric_table']}\n\n"
        )

    user += (
        "Write the final report now. "
        "Use only the template from the system prompt. "
        "Do not diagnose. Do not recommend treatment. "
        "Use cautious clinical language."
    )

    report = chat_node(
        "report_writer",
        system_prompt,
        user,
    )

    allowed_pmids = {
        str(p)
        for p in state.get("pmid_list", [])
        if p
    }

    report = clinical_safety_clean(report)
    report = strict_pmid_sanitizer(report, allowed_pmids)
    report = ensure_required_sections(report)

    cited = cited_pmids_in_report(report)

    if cited:
        report += "\n\n---\nReferences:\n"
        report += "\n".join(
            f"{i + 1}. PMID {p}"
            for i, p in enumerate(cited)
        )

    return {
        **state,
        "final_report": report,
    }


# =============================================================================
# Block IX — Build graph
# =============================================================================

graph = StateGraph(PipelineState)

graph.add_node("data_summariser", data_summariser)
graph.add_node("construct_mapper", construct_mapper)
graph.add_node("pubmed_query_planner", pubmed_query_planner)
graph.add_node("pubmed_retriever", pubmed_retriever)
graph.add_node("relevance_judge", relevance_judge)
graph.add_node("literature_synthesiser", literature_synthesiser)
graph.add_node("symptom_metric_linker", symptom_metric_linker)
graph.add_node("report_writer", report_writer)

graph.set_entry_point("data_summariser")

graph.add_edge("data_summariser", "construct_mapper")
graph.add_edge("construct_mapper", "pubmed_query_planner")
graph.add_edge("pubmed_query_planner", "pubmed_retriever")
graph.add_edge("pubmed_retriever", "relevance_judge")
graph.add_edge("relevance_judge", "literature_synthesiser")


def route_after_synthesis(state: PipelineState) -> str:
    if state.get("anamnesis", "").strip():
        return "symptom_metric_linker"

    return "report_writer"


graph.add_conditional_edges(
    "literature_synthesiser",
    route_after_synthesis,
    {
        "symptom_metric_linker": "symptom_metric_linker",
        "report_writer": "report_writer",
    },
)

graph.add_edge("symptom_metric_linker", "report_writer")
graph.add_edge("report_writer", END)

AGENT_GRAPH = graph.compile()


# =============================================================================
# Block X — Execute
# =============================================================================

print("\nRunning v5.1 Safe Master Pipeline...")

t0 = time.time()

final_state = AGENT_GRAPH.invoke(
    {
        "compact_report": compact_report,
        "audience": AUDIENCE,
        "anamnesis": ANAMNESIS.strip(),
    }
)

elapsed = time.time() - t0

print(f"\nGraph finished in {elapsed:.1f}s")

print("\n# Final v5.1 Safe Report\n")
print(final_state.get("final_report", ""))

print("\n# Debug: Safe Constructs\n")
print(
    json.dumps(
        final_state.get("constructs", []),
        ensure_ascii=False,
        indent=2,
    )
)

print("\n# Debug: Used PMIDs\n")
print(final_state.get("pmid_list", []))

Imports OK
Project root: /Users/arahjou/Documents/APP_CIRCADIAN_MEDICINE_v7

Selected: username=admin P1=ID-001 P2=ID-002

Running v5.1 Safe Master Pipeline...

Detected safe constructs:
- M10: 43.73 → 95.23 (increased, relative_change=1.18) → higher activity during most active period
- Mesor: 22.87 → 48.9 (increased, relative_change=1.14) → higher rhythm-adjusted mean level
- L5: 0.67 → 0.48 (decreased, relative_change=-0.28) → lower activity during least active period
- Sleep Duration: 269.0 → 335.0 (increased, relative_change=0.25) → longer sleep duration
- CPD mid sleep: 0.33 → 0.26 (decreased, relative_change=-0.21) → greater alignment of mid-sleep timing

Deterministic PubMed queries:
- daytime activity rest activity rhythm adults
- cosinor mesor rest activity rhythm adults
- L5 actigraphy rest activity rhythm adults
- long sleep duration health outcomes adults
- sleep timing regularity circadian alignment adults

Graph finished in 248.4s

# Final v5.1 Safe Report

## Bottom Line